# Lesson 0: Environment Setup & Backend Choice
**Goal of this notebook:** by the end, you will have (1) a verified working environment, and (2) a working *backend*, either real IBM quantum hardware (Path A), a local simulator (Path B), or ideally both. Every course notebook assumes this one ran successfully.

**Choose your path:**
- **Path A: Real hardware + simulation (recommended).** Free IBM account, API key saved locally, and a first run on a real quantum computer today. Used in the Lessons 4, 8, and 12 hardware runs.
- **Path B: Simulation only.** Everything runs locally on your own machine and is ready the moment the packages are installed. This path covers the full course content; section 6 explains what simulation gives you and where it differs from real hardware.

You can start with Path B today and add Path A any time later. Nothing else in the course changes.

**Code style promise (applies to every notebook in this course):** samples favor *understanding* over efficiency or cleverness. Explicit loops instead of compact one-liners, named intermediate variables, one idea per cell, and comments that explain *why*. When you later write production code you can compress; while learning, every step stays visible.


---
# 1. Install and verify the environment

Two ways to install the dependencies; pick the one that matches how you got here:

- **Cloned the repository?** Install from `pyproject.toml` in a terminal: `pip install -e .`
  (or `uv sync` if you use uv), then skip the install cell below.
- **Working standalone** (a fresh environment, or Google Colab)? Run the install cell below;
  it installs the same set directly into the kernel this notebook is running on.

What each package is for:
- `qiskit`: the core, circuits, gates, `Statevector`
- `qiskit-aer`: high-performance local simulators (Path B lives here)
- `qiskit-ibm-runtime`: the connection to real IBM hardware (Path A lives here)
- `pylatexenc`: needed for pretty circuit drawings


In [ ]:
# ═══ 1. INSTALL: run once if you have not installed the packages yet ═══
# %pip installs into the exact Python environment this notebook is running on,
# which is why it is preferred over plain !pip inside notebooks.
# Safe to re-run: already-installed packages are skipped.

%pip install qiskit qiskit-aer qiskit-ibm-runtime matplotlib pylatexenc

# If anything was installed just now, restart the kernel once
# (menu: Kernel -> Restart) so the fresh packages load, then continue below.

Now run the smoke test. **Every check must print ✅ before you continue.**

In [ ]:
# ═══ 1. VERIFY: environment smoke test ═══
import sys
print(f"Python {sys.version.split()[0]}  (see pyproject.toml for the required version)")

import numpy as np
import matplotlib
import qiskit
print(f"Qiskit {qiskit.__version__}  (need 1.0+; course examples assume modern primitives)")

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# The 'hello world' of quantum computing: a Bell state.
# (Don't worry about understanding this yet - Lessons 1-2 explain everything.
#  Today it is only a test that the machinery works.)
qc = QuantumCircuit(2)
qc.h(0)        # Hadamard gate on qubit 0
qc.cx(0, 1)    # CNOT gate: qubit 0 controls, qubit 1 is the target

sv = Statevector.from_instruction(qc)

# The 4 entries of the result are the amplitudes of |00>, |01>, |10>, |11>, in that order.
# A Bell state has amplitude 1/sqrt(2) on |00> and |11>, and 0 on the other two:
expected = np.array([1, 0, 0, 1]) / np.sqrt(2)

ok = np.allclose(sv.data, expected)
print(f"{'✅' if ok else '❌'} Bell state statevector correct: {np.round(sv.data, 4)}")

try:
    import qiskit_aer
    print(f"✅ qiskit-aer {qiskit_aer.__version__} (local simulators available)")
except ImportError:
    print("❌ qiskit-aer missing, Path B needs it:  pip install qiskit-aer")

try:
    import qiskit_ibm_runtime
    print(f"✅ qiskit-ibm-runtime {qiskit_ibm_runtime.__version__} (hardware access available)")
except ImportError:
    print("⚠️ qiskit-ibm-runtime missing, only needed for Path A:  pip install qiskit-ibm-runtime")

# Last check: circuit drawing. This needs matplotlib and pylatexenc.
try:
    figure = qc.draw('mpl')
    print("✅ circuit drawing works (a diagram should render below)")
except Exception as error:
    figure = None
    print(f"❌ circuit drawing failed: {error}")
    print("   Usually fixed by: pip install pylatexenc")
figure

---
# 2. Self-test: can you write Python from memory? (two minutes)

This course assumes you can write basic Python and NumPy without looking things up, because
the study method is building from memory. The environment check above tested your machine;
this cell tests you. In the empty cell below, from memory:

1. Write a function that returns the sum of a list of numbers, using an explicit loop.
2. Create a NumPy array from `[1, 2, 3]` and double every entry.
3. Multiply two 2×2 matrices with the `@` operator and print the result.

If all three flowed without friction, you are ready. If any of them required searching, pause
here and spend a few days with a Python fundamentals resource first (the official
[Python tutorial](https://docs.python.org/3/tutorial/) and the
[NumPy quickstart](https://numpy.org/doc/stable/user/quickstart.html) are free and excellent),
then retake this self-test. Starting the course under-equipped turns every lesson into two
struggles at once; starting fluent lets you spend all your effort on the quantum ideas,
which is where it belongs.


In [ ]:
# ═══ 2. your self-test (from memory, no searching) ═══
# 1. A function that returns the sum of a list of numbers, using an explicit loop.

# 2. A NumPy array from [1, 2, 3], with every entry doubled.

# 3. Two 2x2 matrices multiplied with the @ operator, result printed.



---
# 3. Path A: connect to real IBM quantum hardware

**On Path B today?** Skip from here to section 5; sections 3 and 4 will be here whenever you
want them.

**On Path A, do this today or later, your choice.** Nothing in Week 0 through Lesson 3 needs hardware;
the first required run is in Lesson 4. Doing it today has two rewards: any account problems
surface now, while there is slack to fix them, and you get to end your very first session by
running a circuit on a real quantum computer. Skipping it today is equally fine: come back to
this section any time before Lesson 4.

### 3.1 Create the account and get your credentials (in the browser)
1. Sign up free at **https://quantum.cloud.ibm.com** (the IBM Quantum Platform).
2. From the **dashboard**, create an **API key**, copy it immediately to a safe place; *it is shown only once*.
3. From the **Instances** page, hover over your instance's **CRN** and copy it too. (The CRN identifies *which* account instance your jobs bill against; providing it is recommended.)

> ⚠️ **Old tutorials warning:** IBM migrated platforms in 2025. Any tutorial using `channel="ibm_quantum"` with the old quantum-computing.ibm.com site is obsolete. The current channel is `ibm_quantum_platform`, and credentials are an IBM Cloud API key + instance CRN.

### 3.2 Save credentials ONCE, the safe way
The cell below uses `getpass` so your key is **never typed into a notebook cell** (notebooks get committed to git, shared with students, posted in issues... a pasted key *will* eventually leak).
`save_account` writes the credentials to `~/.qiskit/qiskit-ibm.json` on your machine; after this one run, every future notebook connects with zero arguments.


In [ ]:
# ═══ 3.2 RUN ONCE: save credentials (Path A only) ═══
from getpass import getpass
from qiskit_ibm_runtime import QiskitRuntimeService

# Step 1: ask for the credentials. getpass hides what you type,
# so the secret never appears on screen or in the saved notebook.
token = getpass("Paste your IBM Quantum API key (input hidden): ").strip()
crn = getpass("Paste your instance CRN (input hidden, press Enter to skip): ").strip()

# Step 2: save them to disk (~/.qiskit/qiskit-ibm.json).
# set_as_default=True  -> future notebooks can connect with no arguments.
# overwrite=True       -> re-running this cell replaces old credentials instead of erroring.
if crn == "":
    # No CRN provided: save just the token. Qiskit will look up your instances automatically.
    QiskitRuntimeService.save_account(
        token=token,
        channel="ibm_quantum_platform",
        set_as_default=True,
        overwrite=True,
    )
else:
    # CRN provided: also record which instance to use (recommended - fewer lookups).
    QiskitRuntimeService.save_account(
        token=token,
        channel="ibm_quantum_platform",
        instance=crn,
        set_as_default=True,
        overwrite=True,
    )

# Step 3: remove the secrets from the notebook's memory, so nothing lingers
# in variables that a later cell (or a curious student) could print.
del token
del crn

print("✅ Credentials saved, you never need to run this cell again.")

**Security notes (habits worth keeping for every API key you will ever hold):**
- The saved file `~/.qiskit/qiskit-ibm.json` is **plain text**. Fine on a personal machine; on shared machines prefer environment variables (`QISKIT_IBM_TOKEN`, `QISKIT_IBM_INSTANCE`) set in your shell profile instead of `save_account`.
- Never hardcode the key in a cell, and clear cell outputs before committing notebooks (`jupyter nbconvert --clear-output`). Add a `.gitignore` habit for scratch files.
- If a key leaks: delete it from the IBM dashboard and create a new one. Keys are free; leaked compute time is not.


In [ ]:
# ═══ 3.3 VERIFY: connect and list real quantum computers ═══
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()  # no arguments, reads the saved credentials
backends = service.backends(operational=True, simulator=False)
print(f"✅ Connected. {len(backends)} real quantum systems visible to your account:")
for b in backends:
    print(f"   {b.name:<20} {b.num_qubits:>4} qubits   pending jobs: {b.status().pending_jobs}")

lb = service.least_busy(operational=True, simulator=False)
print(f"\nShortest queue right now: {lb.name}")

> **About the free (Open) plan:** it includes a limited allowance of QPU execution time per month, jobs wait in a shared queue (minutes to hours), and unused time doesn't roll over. That's plenty for this course, the hardware runs in Lessons 4, 8, and 12 are seconds of QPU time each. Check your dashboard for your current allowance and usage. **Habit to build now:** debug on the simulator, and send a job to hardware only when the simulated version already works.


---
# 4. Your first run on a real quantum computer (two acts)

**You are not expected to understand this. Not one line of it.** You are running it for one
reason: to see, with your own eyes, a real quantum computer do something no explanation has
earned yet. Your only job is to *observe* and write down what you notice. Resist the urge to
explain; every explanation you could reach for today is wrong in an interesting way.

Here is the plan, three steps:

1. **Rehearsal.** You build two tiny quantum programs and run them on a perfect simulator on
   your own machine, so you know what *should* happen.
2. **Submission.** You send both programs to a real quantum computer in an IBM lab and get a
   ticket number while they wait in line.
3. **Results.** You collect the answers, compare them to the rehearsal, and write down what
   you saw.

The two programs differ by exactly one thing:

- **Act 1** applies one operation (called H) to a qubit and measures it, 1000 times.
- **Act 2** applies the same operation TWICE, then measures, 1000 times.

**Before running anything, write a prediction here:** if one H produces some result, what
should two in a row do?

> My prediction: ...


### 4.1 The rehearsal, on a perfect simulator

Run the next cell. It builds both programs and executes each 1000 times on a flawless
simulator. **What you will see:** two lines, one per act. Each shows a *counts dictionary*
like `{'0': 512, '1': 488}`, which reads: out of 1000 runs, the outcome was `0` in 512 of
them and `1` in 488. Counts are the native language of quantum computers; you will read
thousands of these over the course, and this is your first.

**What to watch for:** compare Act 1's counts to Act 2's, and both to your prediction.
Take your time before moving on; the pause is part of the exercise.


In [ ]:
# ═══ 4. step 1: build the two circuits and get the IDEAL answer from a simulator ═══
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# Act 1: one H, then measure
act1 = QuantumCircuit(1)
act1.h(0)
act1.measure_all()

# Act 2: H twice, then measure
act2 = QuantumCircuit(1)
act2.h(0)
act2.h(0)
act2.measure_all()

# A perfect (noiseless) simulator tells us what SHOULD happen
simulator = AerSimulator()
for name, circuit in [("Act 1 (one H)", act1), ("Act 2 (two Hs)", act2)]:
    result = simulator.run(transpile(circuit, simulator), shots=1000).result()
    print(f"{name}, ideal counts: {result.get_counts()}")

**Pause here.** Act 1 came out close to half-and-half, like a fair coin. And Act 2? Whatever
you predicted, sit with what actually happened for a moment, and jot your reaction below.
No explaining yet; that is Lesson 1's job.

> What I noticed in the rehearsal: ...

### 4.2 Send both programs to a real machine

Now the same two programs go to an actual quantum computer, a physical device near absolute
zero in an IBM lab. Run the next cell. **What you will see, in order:**

- The **name of the machine** chosen for you (the one with the shortest line right now) and
  its qubit count. That name refers to real hardware you could look up on your IBM dashboard.
- A **job id**: your ticket number. Jobs from everyone in the world share these machines, so
  yours waits in a queue: minutes to hours depending on traffic.

**What to do while you wait:** nothing here requires watching. This is the perfect moment to
start `warmup0a_sets_functions_bits.ipynb`; the results will be waiting when you come back.
You can also watch your job's progress on the Workloads page of your IBM dashboard.

**If the cell fails instead:** the most common cause is missing credentials, fixed by
returning to section 3.2. A long queue is normal and needs no fixing.


In [ ]:
# ═══ 4. step 2: submit both circuits to a real quantum computer ═══
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

service = QiskitRuntimeService()                                  # uses your saved credentials
backend = service.least_busy(operational=True, simulator=False)   # shortest queue right now
print(f"Sending both circuits to: {backend.name} ({backend.num_qubits} qubits)")

# Real devices only run circuits adapted to their native gates: that is what transpile does
act1_for_device = transpile(act1, backend)
act2_for_device = transpile(act2, backend)

sampler = SamplerV2(mode=backend)
job = sampler.run([act1_for_device, act2_for_device], shots=1000)
print(f"Job submitted. Ticket (job id): {job.job_id()}")
print("Your circuits are now in line. Go start the first warm-up; come back for step 3.")

In [ ]:
# ═══ 4. optional: check on your job any time ═══
# Run this cell whenever you like. When it says DONE, step 3 is ready.
print(f"Job status: {job.status()}")

### 4.3 Collect the results

When the status above says DONE, run the next cell. **What you will see:** the same two
counts dictionaries as the rehearsal, but this time every number came from a physical qubit,
followed by a bar chart (a *histogram*) showing both acts side by side: outcomes along the
bottom, how often each occurred as the height of its bar.

**What to watch for, three things:**

1. **Act 1 on hardware vs. Act 1 in rehearsal.** Close? They should be.
2. **Act 2 on hardware vs. Act 2 in rehearsal.** Mostly the same story, with one difference:
   look for a small bar that the rehearsal said should not exist at all.
3. **That small bar.** The perfect simulator gave a clean answer; the real machine almost,
   but not quite, agrees. Remember it. It is the most important little bar in this course.

Save the histogram (right-click it, or screenshot): you will want to look back at it during
Lesson 1.


In [ ]:
# ═══ 4. step 3: collect your results (run this after the job finishes) ═══
from qiskit.visualization import plot_histogram

result = job.result()
counts_act1 = result[0].data.meas.get_counts()
counts_act2 = result[1].data.meas.get_counts()

print("Act 1 (one H), real hardware: ", counts_act1)
print("Act 2 (two Hs), real hardware:", counts_act2)

# Save this figure: you will look back at it in Lesson 1.
plot_histogram([counts_act1, counts_act2], legend=["Act 1: one H", "Act 2: two Hs"])

## 📝 Day-one observation log (observations only, no explanations)

- Act 1 gave me roughly: ...
- My prediction for Act 2 was: ...
- Act 2 actually gave me: ...
- Compared to the rehearsal, the real machine differed by: ...
- The strangest part, in one sentence: ...

Three things you just witnessed, stated without explanation. One H looks like a fair coin
flip. The same "coin flip" done twice gives (almost) always 0: whatever H does, doing it
twice undoes it, which no coin can do. And the real machine disagrees slightly with the
perfect simulator: that small extra bar in Act 2 is your first sighting of *noise*.

Lesson 1 pays the debt for Acts 1 and 2. The noise gets a name in Unit III, and Unit IV is
about defeating it. Welcome to the course.

### Optional Act 3, for the curious (preview of Lesson 2)
Add a second qubit: `qc.h(0)` then `qc.cx(0, 1)`, measure both. On hardware you will see
almost only `00` and `11`: two qubits that agree every single time, though nobody assigned
them a value. That trick is called entanglement, it is Lesson 2's subject, and by Lesson 4
you will use it to do things that should be impossible.

*Path B note: the same three steps work without an account by swapping the real backend for
`FakeManilaV2` (section 5.3 shows how). You get the noise, though admittedly some of the
goosebumps require the real machine.*


---
# 5. Path B: local simulation, entirely on your own machine

Three local tools cover everything, and **you will use them constantly even if you have
hardware access**: simulators are where all development happens.


In [ ]:
# ═══ 5.1 Exact statevector: the microscope ═══
# Perfect, noiseless, and you can inspect the full quantum state (impossible on real hardware!).
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

sv = Statevector.from_instruction(qc)
print("Full state (cheating, nature never shows you this):", np.round(sv.data, 4))
print("Measurement probabilities:", sv.probabilities_dict())

In [ ]:
# ═══ 5.2 AerSimulator: the hardware stand-in ═══
# Runs shots like a real device: you get counts, not the state. This is your default backend for Path B.
from qiskit_aer import AerSimulator
from qiskit import transpile

qc_meas = qc.copy()
qc_meas.measure_all()

sim = AerSimulator()
result = sim.run(transpile(qc_meas, sim), shots=1000).result()
print("Counts from 1000 shots:", result.get_counts())
# Expect roughly {'00': ~500, '11': ~500}, and NEVER '01' or '10'. That's entanglement.

In [ ]:
# ═══ 5.3 Noisy simulation: a fake quantum computer ═══
# Snapshots of real IBM devices (calibration data included) let you simulate realistic noise offline.
# This is how Path B users do the 'hardware' portions of Lessons 4, 8, and 12.
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

fake = FakeManilaV2()   # a snapshot of a real 5-qubit device
result = fake.run(transpile(qc_meas, fake), shots=1000).result()
print("Counts on the FAKE noisy device:", result.get_counts())
# Now you WILL see some '01' and '10', noise! Compare with the perfect AerSimulator counts above.

---
# 6. What simulation gives you, and where it differs from real hardware

### Limitation 1: exponential memory, the wall is real
An n-qubit state is 2ⁿ complex amplitudes (16 bytes each). Nothing negotiates with that exponent, run the cell below and find where YOUR machine dies. (For this course it's irrelevant: nothing here exceeds ~10 qubits. It becomes the whole story if you later simulate serious algorithms, which is, of course, exactly why quantum computers are interesting.)


In [ ]:
# ═══ 6. BUILD: the exponential wall, visualized ═══
import matplotlib.pyplot as plt

# The reasoning, step by step:
#   - a state of n qubits is a vector with 2^n complex amplitudes
#   - each complex amplitude is two 64-bit floats = 16 bytes
#   - so total memory = 2^n * 16 bytes
BYTES_PER_AMPLITUDE = 16
BYTES_PER_GIB = 2**30          # 1 GiB = 2^30 bytes

qubit_counts = []
memory_in_gib = []
for n in range(1, 51):
    number_of_amplitudes = 2**n
    total_bytes = number_of_amplitudes * BYTES_PER_AMPLITUDE
    total_gib = total_bytes / BYTES_PER_GIB
    qubit_counts.append(n)
    memory_in_gib.append(total_gib)

# Plot it (log scale on y, because the growth is exponential)
plt.figure(figsize=(8, 4.5))
plt.semilogy(qubit_counts, memory_in_gib, marker='.')

# Draw reference lines for real machines, one at a time
plt.axhline(16, linestyle='--', linewidth=0.8)          # a typical laptop: 16 GiB
plt.text(1, 16 * 1.5, "laptop (16 GiB)", fontsize=8)

plt.axhline(1024, linestyle='--', linewidth=0.8)        # a big server: 1 TiB = 1024 GiB
plt.text(1, 1024 * 1.5, "big server (1 TiB)", fontsize=8)

plt.axhline(1e9, linestyle='--', linewidth=0.8)         # ~all the RAM on Earth (order of magnitude)
plt.text(1, 1e9 * 1.5, "roughly all RAM on Earth", fontsize=8)

plt.xlabel("number of qubits")
plt.ylabel("memory needed for the statevector (GiB, log scale)")
plt.title("Why simulators hit a wall around 30-45 qubits")
plt.tight_layout()
plt.show()

# Print a few landmark values to make it concrete
for n in [20, 30, 40, 50]:
    total_gib = (2**n * BYTES_PER_AMPLITUDE) / BYTES_PER_GIB
    print(f"{n} qubits -> {total_gib:,.1f} GiB just to STORE the state (before doing any computation)")

### Limitation 2: simulated noise is a model, not the messy truth
Fake backends replay a *calibration snapshot*. Real devices drift hour to hour, have readout errors, crosstalk, and occasional surprises no model captures. If you only ever simulate, quantum computing feels cleaner than it is, one real hardware run teaches a humility that no simulator can.

### Limitation 3: the statevector simulator lets you cheat
`Statevector` shows you all amplitudes at once. Real quantum mechanics never does, you get one measurement outcome per shot, full stop. This "god view" is a *fantastic learning tool* (the course notebooks exploit it constantly) but be conscious you're cheating: any reasoning that requires seeing the state is reasoning a real quantum computer can't do. Good self-check when designing algorithms: *could I still do this with counts only?*

### Limitation 4: real devices have queues, connectivity limits, and native gate sets
Real devices have limited qubit connectivity (your CNOT between distant qubits silently becomes a chain of SWAPs), restricted native gate sets, and shared queues. Simulation hides all of this. Lessons 4/8/12 hardware runs exist precisely to surface it.

### Bottom line
| | Path B (simulation only) | Path A (+ hardware) |
|---|---|---|
| Course concepts & all 16 lessons | ✅ fully | ✅ fully |
| Noise experience | 🟡 modeled (fake backends) | ✅ the real thing |
| Cost / setup | ✅ zero | ✅ free tier, small setup |
| "I ran this on an actual quantum computer" | ❌ | ✅ (priceless for motivation & teaching) |

**Recommendation:** develop everything on simulators regardless of path; if you can, add Path A for the three hardware touchpoints. For classrooms without accounts, Path B + fake backends is a legitimate, complete course experience.


---
# 7. 🎓 Wrap-up: final readiness check

Run the cell below. It detects your setup and tells you exactly where you stand.


In [ ]:
# ═══ Readiness check ═══
# Check 1: is the Qiskit core installed? (required for everything)
try:
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector
    core_works = True
except ImportError:
    core_works = False

if core_works:
    print("✅ Qiskit core          -> required for everything")
else:
    print("❌ Qiskit core missing  -> run: pip install qiskit")

# Check 2: is the local simulator installed? (this is Path B)
try:
    from qiskit_aer import AerSimulator
    simulation_works = True
except ImportError:
    simulation_works = False

if simulation_works:
    print("✅ Local simulation     -> Path B ready")
else:
    print("❌ Local simulation missing -> run: pip install qiskit-aer")

# Check 3: can we reach real IBM hardware? (this is Path A)
# Note: this one can fail for several reasons (package not installed,
# no saved credentials, no internet), so we catch ANY exception,
# and failing is perfectly fine - Path B covers the whole course.
try:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService()   # reads saved credentials from disk
    service.backends()                 # actually talks to IBM to prove it works
    hardware_works = True
except Exception:
    hardware_works = False

if hardware_works:
    print("✅ IBM hardware access  -> Path A ready")
else:
    print("⚪ IBM hardware access  -> Path A not configured (fine: Path B covers the course)")

if core_works and simulation_works:
    print()
    print("🚀 You are ready. Next stop: warmup0a_sets_functions_bits.ipynb")